In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

CATALOG_SCHEMA = "workspace.ecommerce_de"
RAW_BASE = "/Volumes/workspace/ecommerce_de/raw_landing"
TRACKER_TABLE = f"{CATALOG_SCHEMA}.bronze_ingested_files"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TRACKER_TABLE} (
    source_name STRING,
    file_path STRING,
    ingested_at TIMESTAMP
) USING DELTA
""")


def get_new_files(source_folder: str) -> list[str]:
    folder_path = f"{RAW_BASE}/{source_folder}"
    all_files = [f.path for f in dbutils.fs.ls(folder_path) if not f.isDir()]

    processed_df = spark.table(TRACKER_TABLE).filter(F.col("source_name") == source_folder)
    processed_set = {row["file_path"] for row in processed_df.select("file_path").collect()}

    return [p for p in all_files if p not in processed_set]


def mark_files_processed(source_folder: str, file_paths: list[str]):
    if not file_paths:
        return
    rows = [(source_folder, p) for p in file_paths]
    df_new = spark.createDataFrame(rows, ["source_name", "file_path"]).withColumn(
        "ingested_at", F.current_timestamp()
    )
    df_new.write.format("delta").mode("append").saveAsTable(TRACKER_TABLE)


def ingest_source(source_folder: str, target_table: str, file_format: str):
    new_files = get_new_files(source_folder)
    if not new_files:
        print(f"⏭️ No new files for {source_folder} -> {target_table}")
        return

    if file_format == "csv":
        df = spark.read.option("header", "true").option("mergeSchema", "true").csv(new_files)
    elif file_format == "json":
        df = spark.read.option("mergeSchema", "true").json(new_files)
    else:
        raise ValueError(f"Unsupported format: {file_format}")

    # Keep raw values as STRING in Bronze to preserve raw fidelity & handle schema drift
    df_bronze = (
        df.select([F.col(c).cast(StringType()).alias(c) for c in df.columns])
        .withColumn("source_name", F.lit(source_folder))
        .withColumn("source_file_path", F.col("_metadata.file_path"))
        .withColumn("bronze_ingested_at", F.current_timestamp())
    )

    df_bronze.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(
        target_table
    )
    mark_files_processed(source_folder, new_files)
    print(f"✅ Ingested {len(new_files)} file(s) into {target_table}")


ingest_source("orders_csv", f"{CATALOG_SCHEMA}.bronze_orders", "csv")
ingest_source("payments_json", f"{CATALOG_SCHEMA}.bronze_payments", "json")
ingest_source("customers_snapshot", f"{CATALOG_SCHEMA}.bronze_customers", "csv")

In [0]:
%sql
SELECT 'bronze_orders' AS tbl, COUNT(*) AS row_cnt FROM workspace.ecommerce_de.bronze_orders
UNION ALL
SELECT 'bronze_payments', COUNT(*) FROM workspace.ecommerce_de.bronze_payments
UNION ALL
SELECT 'bronze_customers', COUNT(*) FROM workspace.ecommerce_de.bronze_customers;